# Random Forest Experiments
This notebook is a lightweight sandbox for primitive test runs using Random Forest.
Run cells in order from top to bottom.

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
import joblib

BASE_DIR = "datasets_full.csv/datasets_full.csv"
USERS_FILE = "users.csv"

DATASETS = {
    "genuine_accounts.csv": 0,
    "fake_followers.csv": 1,
    "social_spambots_1.csv": 1,
    "social_spambots_2.csv": 1,
    "social_spambots_3.csv": 1,
    "traditional_spambots_1.csv": 1,
    "traditional_spambots_2.csv": 1,
    "traditional_spambots_3.csv": 1,
    "traditional_spambots_4.csv": 1,
}

USER_FEATURES = [
    "statuses_count", "followers_count", "friends_count",
    "favourites_count", "listed_count", "default_profile",
    "default_profile_image", "verified"
]

TWEET_FEATURES = [
   "reply_count", "favorite_count", "num_urls", "num_mentions"
]

RF_PARAMS = {
    "n_estimators": 300,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
}

print("Dataset root:", BASE_DIR)
print("Files configured:", len(DATASETS))
print("Random Forest params:", RF_PARAMS)

Dataset root: datasets_full.csv/datasets_full.csv
Files configured: 9
Random Forest params: {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}


In [2]:
def resolve_users_csv(base_dir, dataset_entry):
    """Resolve dataset entry to a users.csv path."""
    candidates = [
        os.path.join(base_dir, dataset_entry),
        os.path.join(base_dir, dataset_entry, USERS_FILE),
        os.path.join(base_dir, dataset_entry, dataset_entry, USERS_FILE),
    ]
    for path in candidates:
        if os.path.isfile(path):
            return path
    return None


def load_all_data(dataset_map, base_dir):
    """Load all configured datasets and attach binary labels."""
    frames = []
    missing_files = []

    for dataset_entry, label in dataset_map.items():
        path = resolve_users_csv(base_dir, dataset_entry)
        if path is None:
            missing_files.append(dataset_entry)
            continue

        df = pd.read_csv(
            path,
            encoding="utf-8",
            on_bad_lines="skip",
            low_memory=False,
        )
        df["label"] = int(label)
        df["source_file"] = dataset_entry
        frames.append(df)
        print(f"Loaded {dataset_entry}: {len(df):,} rows")

    if missing_files:
        print("\nMissing dataset entries:")
        for entry in missing_files:
            print("-", entry)

    if not frames:
        raise ValueError("No dataset files were loaded. Check BASE_DIR and DATASETS.")

    all_data = pd.concat(frames, ignore_index=True)
    print(f"\nTotal rows loaded: {len(all_data):,}")
    return all_data


def load_tweets(base_dir, dataset_entry):
    """Load and aggregate tweet-level features per user."""
    candidates = [
        os.path.join(base_dir, dataset_entry, "tweets.csv"),
        os.path.join(base_dir, dataset_entry, dataset_entry, "tweets.csv"),
    ]

    for path in candidates:
        if os.path.isfile(path):
            available = ["user_id"] + TWEET_FEATURES
            chunks = []
            for chunk in pd.read_csv(
                path,
                usecols=lambda c: c in available,
                chunksize=100_000,
                encoding="utf-8",
                on_bad_lines="skip",
            ):
                chunks.append(chunk)

            dataframe = pd.concat(chunks, ignore_index=True)
            agg_cols = [c for c in TWEET_FEATURES if c in dataframe.columns]
            dataframe[agg_cols] = dataframe[agg_cols].apply(pd.to_numeric, errors="coerce")
            agg_cols = [c for c in agg_cols if pd.api.types.is_numeric_dtype(dataframe[c])]

            if not agg_cols:
                return None

            return dataframe.groupby("user_id")[agg_cols].mean().reset_index()

    return None

In [3]:
# Block 1: Data loading (full configured datasets)
raw = load_all_data(DATASETS, BASE_DIR)

tweet_aggs = []
for dataset_entry in DATASETS:
    agg = load_tweets(BASE_DIR, dataset_entry)
    if agg is not None:
        tweet_aggs.append(agg)

if tweet_aggs:
    all_tweets = pd.concat(tweet_aggs, ignore_index=True)
    if "id" in raw.columns:
        raw = raw.rename(columns={"id": "user_id"})
    raw = raw.merge(all_tweets, on="user_id", how="left")

print("Raw data shape:", raw.shape)

Loaded genuine_accounts.csv: 3,474 rows
Loaded fake_followers.csv: 3,351 rows
Loaded social_spambots_1.csv: 991 rows
Loaded social_spambots_2.csv: 3,457 rows
Loaded social_spambots_3.csv: 464 rows
Loaded traditional_spambots_1.csv: 1,000 rows
Loaded traditional_spambots_2.csv: 100 rows
Loaded traditional_spambots_3.csv: 403 rows
Loaded traditional_spambots_4.csv: 1,128 rows

Total rows loaded: 14,368
Raw data shape: (14368, 48)


In [4]:
# Block 2: Slicing and preprocessing
available_numeric = [c for c in USER_FEATURES if c in raw.columns]
if not available_numeric:
    raise ValueError("No numeric feature columns found in raw data.")

X = raw[available_numeric].copy()
X = X.apply(pd.to_numeric, errors="coerce")

tweet_features = [c for c in TWEET_FEATURES if c in raw.columns]
X = pd.concat([X, raw[tweet_features]], axis=1)
X = X.fillna(0)

print("Features:", list(X.columns))
print("Shape:", X.shape)

y = raw["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
class_ratio = neg_count / pos_count if pos_count else 1.0

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Class balance:\n", y.value_counts(dropna=False))
print(f"Train class ratio (neg/pos): {class_ratio:.4f}")

Features: ['statuses_count', 'followers_count', 'friends_count', 'favourites_count', 'listed_count', 'default_profile', 'default_profile_image', 'verified', 'reply_count', 'favorite_count', 'num_urls', 'num_mentions']
Shape: (14368, 12)
Train shape: (11494, 12)
Test shape: (2874, 12)
Class balance:
 label
1    10894
0     3474
Name: count, dtype: int64
Train class ratio (neg/pos): 0.3189


In [5]:
# Block 3: Training and cross-validated benchmarking
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline = DummyClassifier(strategy="most_frequent")
baseline_scores = cross_validate(
    baseline,
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

rf_cv_scores = cross_validate(
    RandomForestClassifier(**RF_PARAMS),
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

print("\n5-fold stratified cross-validation")
for metric_name in scoring:
    baseline_values = baseline_scores[f"test_{metric_name}"]
    model_values = rf_cv_scores[f"test_{metric_name}"]
    print(
        f"{metric_name:>18}: baseline {baseline_values.mean():.4f} ± {baseline_values.std():.4f} | "
        f"random forest {model_values.mean():.4f} ± {model_values.std():.4f}"
    )

model = RandomForestClassifier(**RF_PARAMS)
model.fit(X_train, y_train)

print("\nTraining complete.")
print("Number of trees:", len(model.estimators_))
print("OOB score available:", hasattr(model, "oob_score_") and getattr(model, "oob_score_", None) is not None)


5-fold stratified cross-validation
          accuracy: baseline 0.7582 ± 0.0001 | random forest 0.9943 ± 0.0020
 balanced_accuracy: baseline 0.5000 ± 0.0000 | random forest 0.9915 ± 0.0028
          f1_macro: baseline 0.4312 ± 0.0000 | random forest 0.9922 ± 0.0027
           roc_auc: baseline 0.5000 ± 0.0000 | random forest 0.9998 ± 0.0001
 average_precision: baseline 0.7582 ± 0.0001 | random forest 0.9999 ± 0.0000

Training complete.
Number of trees: 300
OOB score available: False


In [6]:
# Block 4: Evaluation
pred = model.predict(X_test)
pred_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, pred),
    "f1_macro": f1_score(y_test, pred, average="macro"),
    "roc_auc": roc_auc_score(y_test, pred_proba),
    "average_precision": average_precision_score(y_test, pred_proba),
}

print("Holdout metrics")
for metric_name, metric_value in metrics.items():
    print(f"{metric_name:>18}: {metric_value:.4f}")

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
print("\nBaseline on same holdout")
print(f"{'accuracy':>18}: {accuracy_score(y_test, baseline_pred):.4f}")
print(f"{'balanced_accuracy':>18}: {balanced_accuracy_score(y_test, baseline_pred):.4f}")
print(f"{'f1_macro':>18}: {f1_score(y_test, baseline_pred, average='macro'):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred))
print("\nClassification Report:")
print(classification_report(y_test, pred, digits=4))

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_,
})
importance_df = importance_df.sort_values("importance", ascending=False)

print("\nTop 10 features by importance:")
display(importance_df.head(10))

if "source_file" in raw.columns:
    test_results = pd.DataFrame(
        {
            "source_file": raw.loc[X_test.index, "source_file"].values,
            "true_label": y_test.values,
            "pred_label": pred,
            "pred_proba": pred_proba,
        },
        index=X_test.index,
    )
    test_results["correct"] = test_results["true_label"] == test_results["pred_label"]
    source_summary = (
        test_results.groupby(["source_file", "true_label"], as_index=False)
        .agg(
            support=("correct", "size"),
            accuracy=("correct", "mean"),
            avg_pred_proba=("pred_proba", "mean"),
            predicted_bot_rate=("pred_label", "mean"),
        )
        .sort_values(["true_label", "accuracy"], ascending=[True, False])
    )
    print("\nPer-source holdout summary:")
    display(source_summary)

Holdout metrics
          accuracy: 0.9937
 balanced_accuracy: 0.9895
          f1_macro: 0.9914
           roc_auc: 0.9998
 average_precision: 0.9999

Baseline on same holdout
          accuracy: 0.7582
 balanced_accuracy: 0.5000
          f1_macro: 0.4312

Confusion Matrix:
[[ 682   13]
 [   5 2174]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9927    0.9813    0.9870       695
           1     0.9941    0.9977    0.9959      2179

    accuracy                         0.9937      2874
   macro avg     0.9934    0.9895    0.9914      2874
weighted avg     0.9937    0.9937    0.9937      2874


Top 10 features by importance:


,feature,importance
3,favourites_count,0.396923
0,statuses_count,0.239297
1,followers_count,0.088896
11,num_mentions,0.086074
9,favorite_count,0.071803
10,num_urls,0.058035
2,friends_count,0.026595
4,listed_count,0.022660
8,reply_count,0.005445
5,default_profile,0.003511



Per-source holdout summary:


,source_file,true_label,support,accuracy,avg_pred_proba,predicted_bot_rate
1,genuine_accounts.csv,0,695,0.981295,0.026432,0.018705
2,social_spambots_1.csv,1,207,1.000000,0.997198,1.000000
3,social_spambots_2.csv,1,681,1.000000,0.999868,1.000000
4,social_spambots_3.csv,1,83,1.000000,0.993695,1.000000
5,traditional_spambots_1.csv,1,210,1.000000,0.997841,1.000000
6,traditional_spambots_2.csv,1,21,1.000000,0.961905,1.000000
8,traditional_spambots_4.csv,1,230,1.000000,0.997377,1.000000
0,fake_followers.csv,1,677,0.998523,0.997410,0.998523
7,traditional_spambots_3.csv,1,70,0.942857,0.881381,0.942857


In [7]:
# Block 5: Feature importance list
importance_sorted = importance_df.sort_values("importance", ascending=False)

print("Top 10 Features — Random Forest:")
for i, row in enumerate(importance_sorted.head(10).itertuples(index=False), 1):
    print(f"  {i:2}. {row.feature:<30} {row.importance:.4f}")

Top 10 Features — Random Forest:
   1. favourites_count               0.3969
   2. statuses_count                 0.2393
   3. followers_count                0.0889
   4. num_mentions                   0.0861
   5. favorite_count                 0.0718
   6. num_urls                       0.0580
   7. friends_count                  0.0266
   8. listed_count                   0.0227
   9. reply_count                    0.0054
  10. default_profile                0.0035


In [8]:
# Block 6: Save model
MODEL_PATH = "random_forest_bot_detector.joblib"
joblib.dump(model, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

Model saved to random_forest_bot_detector.joblib


## Run Order
1. Cell 1: imports and constants
2. Cell 2: helper functions
3. Cell 3: full data loading
4. Cell 4: slicing and preprocessing
5. Cell 5: cross-validated benchmarking and training
6. Cell 6: holdout evaluation and source breakdown
7. Cell 7: feature importance list
8. Cell 8: save model